# 安装 yolov5（若未安装）

In [ ]:
!pip install -q ultralytics

# 导入必要的库

In [ ]:
import torch
from torchvision import datasets, transforms
import os
import cv2
import numpy as np
from tqdm import tqdm

# 1. 准备 MNIST 为 YOLO 格式（每张图一个中心框）

In [ ]:
def mnist_to_yolo(root='mnist_yolo'):
    os.makedirs(f'{root}/images/train', exist_ok=True)
    os.makedirs(f'{root}/labels/train', exist_ok=True)

    transform = transforms.ToTensor()
    mnist = datasets.MNIST('data', train=True, download=True, transform=transform)

    for i, (img_tensor, label) in enumerate(tqdm(mnist)):
        # 转为 0-255 uint8 图像（28x28）
        img = (img_tensor.squeeze().numpy() * 255).astype(np.uint8)
        # 放大到 416x416（YOLO 推荐输入尺寸）
        img_big = cv2.resize(img, (416, 416), interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(f'{root}/images/train/{i}.png', img_big)

        # YOLO 标签: class cx cy w h (归一化)
        # 原图数字居中，假设占满整图 → cx=0.5, cy=0.5, w=1.0, h=1.0
        with open(f'{root}/labels/train/{i}.txt', 'w') as f:
            f.write(f'{label} 0.5 0.5 1.0 1.0\n')

mnist_to_yolo()

# 2. 创建 YOLO 数据配置文件

In [ ]:
yaml_content = """
train: ./mnist_yolo/images/train
val: ./mnist_yolo/images/train

nc: 10
names: ['0','1','2','3','4','5','6','7','8','9']
"""

with open('mnist.yaml', 'w') as f:
    f.write(yaml_content)

# 3. 训练 YOLO 模型

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov5s.pt')
model.train(data='mnist.yaml', epochs=3, imgsz=416, batch=64, name='yolo_mnist')

# 4. 推理示例

In [ ]:
results = model('mnist_yolo/images/train/0.png')
results[0].show()  # 显示带框结果